<a href="https://colab.research.google.com/github/jnrahul92/FastAI_Learn/blob/main/C10_Fast_AI.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
from fastai.text.all import *

In [2]:
path = untar_data(URLs.IMDB)

<div><progress max="144440600" value="144441344"></progress> 100.00% [144441344/144440600 00:04&lt;00:00]</div>

In [3]:
files = get_text_files(path, folders=['train','test','unsup'])

In [4]:
txt = files[0].open().read();

In [5]:
txt[:75]

'Albert Brooks should have let life imitate art and hired himself a muse bef'

In [6]:
spacy = WordTokenizer()
toks = first(spacy([txt]))

In [7]:
print(coll_repr(toks,30))

(#90) ['Albert', 'Brooks', 'should', 'have', 'let', 'life', 'imitate', 'art', 'and', 'hired', 'himself', 'a', 'muse', 'before', 'deciding', 'to', 'torture', 'us', 'with', 'this', 'crap', '.', 'A', 'great', 'cast', '(', 'including', 'the', 'usually', 'much'...]


In [8]:
first(spacy(['The U.S. dollar $1 is $1.00.']))

['The', 'U.S.', 'dollar', '$', '1', 'is', '$', '1.00', '.']

In [9]:
tkn = Tokenizer(spacy)
print(coll_repr(tkn(txt),31))

(#101) ['xxbos', 'xxmaj', 'albert', 'xxmaj', 'brooks', 'should', 'have', 'let', 'life', 'imitate', 'art', 'and', 'hired', 'himself', 'a', 'muse', 'before', 'deciding', 'to', 'torture', 'us', 'with', 'this', 'crap', '.', 'a', 'great', 'cast', '(', 'including', 'the'...]


In [10]:
coll_repr(tkn('&copy; Fast.ai www.fast.ai/INDEX'), 31)

"['xxbos', '©', 'xxmaj', 'fast.ai', 'xxrep', '3', 'w', '.fast.ai', '/', 'xxup', 'index']"

In [11]:
txts = L(o.open().read() for o in files[:2000])

In [12]:
def subword(sz):
    sp = SubwordTokenizer(vocab_sz=sz)
    sp.setup(txts)
    return ' '.join(first(sp([txt]))[:40])

In [13]:
toks = tkn(txt)

In [14]:
toks200 = txts[:200].map(tkn)

In [15]:
num = Numericalize()
num.setup(toks200)

In [17]:
nums200 = toks200.map(num)

In [18]:
dl = LMDataLoader(nums200)

In [19]:
x,y = first(dl)
x.shape, y.shape

(torch.Size([64, 72]), torch.Size([64, 72]))

In [20]:
get_imdb = partial(get_text_files, folders = ['train','test','unsup'])

In [21]:
dls_lm = DataBlock(
    blocks = TextBlock.from_folder(path, is_lm = True),
    get_items= get_imdb, splitter=RandomSplitter(0.1)).dataloaders(path, path = path, bs = 128, seq_len = 80)

In [22]:
dls_lm.show_batch(max_n = 2)

xxbos i saw this at a sneak preview hosted by the xxmaj jewish xxmaj community xxmaj center , and was probably the only xxunk in the audience ! xxmaj this movie deeply moved me . xxmaj robin xxmaj williams deserves an xxmaj oscar for his portrayal of xxmaj jakob . xxmaj the supporting cast was fantastic . i highly recommend it . xxbos i thought it was an excellent movie . xxmaj gary xxmaj cole played the role of a
for 6 months , and this film could just as easily summarize the strange ennui and frustration of any xxmaj asian metropolis as it takes on xxmaj xxunk here . xxmaj it uses the myth of xxmaj hong xxmaj kong musicals the same way xxmaj godard or xxmaj hartley use xxmaj western musicals , but takes it to an extreme , it 's gritty world and occasionally xxmaj kafka - esquire logic make it all the better . i really
i saw this at a sneak preview hosted by the xxmaj jewish xxmaj community xxmaj center , and was probably the only xxunk in the audience ! xxmaj this movie deeply moved 

In [23]:
learn = language_model_learner(
    dls_lm, AWD_LSTM, drop_mult=0.3,
metrics=[accuracy, Perplexity()]).to_fp16()

<div><progress max="105067061" value="105070592"></progress> 100.00% [105070592/105067061 00:03&lt;00:00]</div>

In [24]:
learn.fit_one_cycle(1, 2e-2)

epoch,train_loss,valid_loss,accuracy,perplexity,time
0,4.023647,3.919168,0.299328,50.358505,21:03


In [25]:
learn.save_encoder('finetuned')

In [26]:
TEXT = 'I liked this movie because'

In [27]:
N_WORDS = 40
N_SENTENCES = 2

In [28]:
preds = [learn.predict(TEXT, N_WORDS, temperature=0.75) for _ in range(N_SENTENCES)]

In [29]:
print("\n".join(preds))

i liked this movie because the story was very realistic because the plot , some scenes with the protagonists ' moves , the plot and the storyline and the story line are really great . The story is about the just life of the
i liked this movie because it was different from what i expected the original . This is a great movie but it did n't do it justice , it 's a bad story , but overall it 's a bad movie that is fun


In [30]:
dls_class = DataBlock(blocks = (TextBlock.from_folder(path, vocab=dls_lm.vocab),CategoryBlock),
                      get_y = parent_label,
                      get_items = partial(get_text_files,folders = ['train','test']),
                      splitter = GrandparentSplitter(valid_name='test')).dataloaders(path, path = path, bs = 128, seq_len = 72)

In [32]:
dls_class.show_batch(max_n = 3)

,text,category
0,"xxbos xxmaj match 1 : xxmaj tag xxmaj team xxmaj table xxmaj match xxmaj bubba xxmaj ray and xxmaj spike xxmaj dudley vs xxmaj eddie xxmaj guerrero and xxmaj chris xxmaj benoit xxmaj bubba xxmaj ray and xxmaj spike xxmaj dudley started things off with a xxmaj tag xxmaj team xxmaj table xxmaj match against xxmaj eddie xxmaj guerrero and xxmaj chris xxmaj benoit . xxmaj according to the rules of the match , both opponents have to go through tables in order to get the win . xxmaj benoit and xxmaj guerrero heated up early on by taking turns hammering first xxmaj spike and then xxmaj bubba xxmaj ray . a xxmaj german xxunk by xxmaj benoit to xxmaj bubba took the wind out of the xxmaj dudley brother . xxmaj spike tried to help his brother , but the referee restrained him while xxmaj benoit and xxmaj guerrero",pos
1,"xxbos * * attention xxmaj spoilers * * \n\n xxmaj first of all , let me say that xxmaj rob xxmaj roy is one of the best films of the 90 's . xxmaj it was an amazing achievement for all those involved , especially the acting of xxmaj liam xxmaj neeson , xxmaj jessica xxmaj lange , xxmaj john xxmaj hurt , xxmaj brian xxmaj cox , and xxmaj tim xxmaj roth . xxmaj michael xxmaj canton xxmaj jones painted a wonderful portrait of the honor and dishonor that men can represent in themselves . xxmaj but alas … \n\n it constantly , and unfairly gets compared to "" braveheart "" . xxmaj these are two entirely different films , probably only similar in the fact that they are both about xxmaj scots in historical xxmaj scotland . xxmaj yet , this comparison frequently bothers me because it seems",pos
2,"xxbos xxmaj titanic directed by xxmaj james xxmaj cameron presents a fictional love story on the historical setting of the xxmaj titanic . xxmaj the plot is simple , xxunk , or not for those who love plots that twist and turn and keep you in suspense . xxmaj the end of the movie can be figured out within minutes of the start of the film , but the love story is an interesting one , however . xxmaj kate xxmaj winslett is wonderful as xxmaj rose , an aristocratic young lady betrothed by xxmaj cal ( billy xxmaj zane ) . xxmaj early on the voyage xxmaj rose meets xxmaj jack ( leonardo dicaprio ) , a lower class artist on his way to xxmaj america after winning his ticket aboard xxmaj titanic in a poker game . xxmaj if he wants something , he goes and gets it",pos


In [33]:
learn = text_classifier_learner(dls_class, AWD_LSTM, drop_mult=0.5, metrics=accuracy).to_fp16()

In [34]:
learn.load_encoder('finetuned')

In [35]:
learn.fit_one_cycle(1, 2e-2)

epoch,train_loss,valid_loss,accuracy,time
0,0.377127,0.311731,0.870120,01:22


In [37]:
learn.freeze_to(-2)
learn.fit_one_cycle(1, slice(1e-2/(2.6**4),1e-2))

epoch,train_loss,valid_loss,accuracy,time
0,0.305467,0.244450,0.903600,01:21


In [38]:
learn.freeze_to(-3)
learn.fit_one_cycle(1, slice(5e-3/(2.6**4),5e-3))

epoch,train_loss,valid_loss,accuracy,time
0,0.240414,0.196929,0.923480,01:45


In [39]:
learn.unfreeze()
learn.fit_one_cycle(2, slice(1e-3/(2.6**4),1e-3))

epoch,train_loss,valid_loss,accuracy,time
0,0.204147,0.195376,0.924600,02:08
1,0.184602,0.180884,0.929400,02:07
